In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('./data/premium.csv')
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1333 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


In [4]:
# bmi의 null 처리
df['bmi'] = df['bmi'].fillna(df['bmi'].mean())
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [5]:
# 문자열 데이터의 수치화 > LabelEncoder
from sklearn.preprocessing import LabelEncoder

In [7]:
col_list = ['sex', 'smoker','region']
for col in col_list :
  enc=LabelEncoder()
  df[col] = enc.fit_transform(df[col])

df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,0,27.900,0,1,3,16884.92400
1,18,1,33.770,1,0,2,1725.55230
2,28,1,33.000,3,0,2,4449.46200
3,33,1,22.705,0,0,1,21984.47061
4,32,1,28.880,0,0,1,3866.85520


In [10]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

In [11]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [12]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [13]:
# 스케일링
num_cols = ['bmi']
#num_cols

X_train[num_cols] = scaler.fit_transform(X_train[num_cols]) # 훈련은 fit도 포함
X_test[num_cols] = scaler.transform(X_test[num_cols]) # 테스트는 transform만

display(X_train[num_cols].describe().round(4))

display(X_test[num_cols].describe().round(4))

,bmi
count,1070.0000
mean,0.0000
std,1.0005
min,-2.4246
25%,-0.7201
50%,-0.0506
75%,0.6439
max,3.7506


,bmi
count,268.0000
mean,0.0864
std,1.0478
min,-2.2826
25%,-0.6451
50%,0.0375
75%,0.7718
max,3.6592


In [25]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 선형회귀 

In [32]:
from sklearn.linear_model import LinearRegression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test,y_pred)
mae, mse, rmse, r2

(4172.783193083617,
 33640928.3211508,
 np.float64(5800.0800271333155),
 0.7833094802263592)

# 다항회귀

In [33]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

In [35]:
degree=[2,3,4]

for deg in degree:
  model_poly = Pipeline([
    ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
    ('linear', LinearRegression())])
  model_poly.fit(X_train, y_train)
  poly_pred = model_poly.predict(X_test)

  mae = mean_absolute_error(y_test, poly_pred)
  mse = mean_squared_error(y_test, poly_pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test, poly_pred)

  print(f'Degree: {deg} MAE: {mae:.4f} MSE: {mae:.4f} R^2: {r2:.4f}')


Degree: 2 MAE: 2740.6271 MSE: 2740.6271 R^2: 0.8664
Degree: 3 MAE: 2818.6301 MSE: 2818.6301 R^2: 0.8610
Degree: 4 MAE: 3120.6621 MSE: 3120.6621 R^2: 0.7786


In [38]:
from sklearn.ensemble import RandomForestRegressor
model_rf = RandomForestRegressor(n_estimators=100, random_state=0)
model_rf.fit(X_train, y_train)
y_pred_rf = lr_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred_rf)
mse = mean_squared_error(y_test, y_pred_rf)
rmse = np.sqrt(mse)
r2 = r2_score(y_test,y_pred_rf)
mae, mse, rmse, r2

(4172.783193083617,
 33640928.3211508,
 np.float64(5800.0800271333155),
 0.7833094802263592)

In [39]:
model_rf.feature_importances_

array([0.13626757, 0.00692881, 0.2079445 , 0.02268396, 0.61074257,
       0.01543259])

In [45]:
# 특성 중요도
import pandas as pd
feature_names = ['age','sex','bmi','children','smoker', 'region']
importance_df = pd.DataFrame({
  'feature': feature_names,
  'importance':model_rf.feature_importances_
}).sort_values('importance',ascending=False)

print(f'특성 중요도 \n------------------------ \n {importance_df}')


특성 중요도 
------------------------ 
     feature  importance
4    smoker    0.610743
2       bmi    0.207945
0       age    0.136268
3  children    0.022684
5    region    0.015433
1       sex    0.006929


In [ ]:
# XGBRegressor